In [1]:
!pip install -q datasets scikit-learn pandas joblib

In [15]:
import json
import re
from pathlib import Path

import shutil
import textwrap

import joblib
import pandas as pd

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import make_scorer, f1_score,accuracy_score, precision_recall_fscore_support

from imdb_sentiment.data.dataset import load_imdb_dataset
from imdb_sentiment.models.baseline import build_baseline_model

from google.colab import files

In [3]:
!git clone https://github.com/Nusultan11/imdb-sentiment.git
%cd /content/imdb-sentiment
!pip install -q -e .

fatal: destination path 'imdb-sentiment' already exists and is not an empty directory.
/content/imdb-sentiment
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for imdb-sentiment (pyproject.toml) ... done


In [4]:
baseline_path = Path("artifacts/reports/val_metrics.json")

print("exists:", baseline_path.exists())

if baseline_path.exists():
    baseline_metrics = json.loads(baseline_path.read_text(encoding="utf-8"))
    print(baseline_metrics)

exists: True
{'accuracy': 0.8976, 'precision': 0.8994391025641025, 'recall': 0.8958499600957701, 'f1': 0.897640943622551}


In [6]:
dataset = load_imdb_dataset()
full_train = dataset["train"]

texts = list(full_train["text"])
labels = list(full_train["label"])

x_train, x_val, y_train, y_val = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
)

model = build_baseline_model(
    max_features=20000,
    ngram_range=(1, 2),
    max_iter=1000,
    random_state=42,
)

param_grid = {
    "vectorizer__max_features": [20000, 25000, 30000, 35000],
    "classifier__C": [0.5, 1.0, 2.0],
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=make_scorer(f1_score),
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid.fit(x_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best params: {'classifier__C': 2.0, 'vectorizer__max_features': 35000}
Best CV F1: 0.8938794184973163


In [8]:
best_model = grid.best_estimator_

y_val_pred = best_model.predict(x_val)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_val,
    y_val_pred,
    average="binary",
    pos_label=1,
    zero_division=0,
)

tuned_val_metrics = {
    "accuracy": float(accuracy_score(y_val, y_val_pred)),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
}

print("Baseline val metrics:", baseline_metrics)
print("Tuned val metrics:", tuned_val_metrics)
print("Best params:", grid.best_params_)

Baseline val metrics: {'accuracy': 0.8976, 'precision': 0.8994391025641025, 'recall': 0.8958499600957701, 'f1': 0.897640943622551}
Tuned val metrics: {'accuracy': 0.8994, 'precision': 0.8945063694267515, 'recall': 0.9042253521126761, 'f1': 0.8993396037622573}
Best params: {'classifier__C': 2.0, 'vectorizer__max_features': 35000}


In [9]:
tuning_dir = Path("artifacts/tuning/tfidf_narrow_v1")
tuning_dir.mkdir(parents=True, exist_ok=True)

# 1. сохраняем лучшую модель
joblib.dump(best_model, tuning_dir / "model.joblib")

# 2. сохраняем лучшие параметры
best_params_payload = {
    "best_params": grid.best_params_,
    "best_cv_f1": float(grid.best_score_),
}
(tuning_dir / "best_params.json").write_text(
    json.dumps(best_params_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# 3. сохраняем val-метрики tuned модели
(tuning_dir / "val_metrics.json").write_text(
    json.dumps(tuned_val_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# 4. сохраняем сравнение с baseline
comparison = {
    "baseline": baseline_metrics,
    "tuned_model": tuned_val_metrics,
    "best_params": grid.best_params_,
    "delta_tuned_minus_baseline": {
        metric: round(tuned_val_metrics[metric] - baseline_metrics[metric], 6)
        for metric in baseline_metrics.keys()
    },
    "winner_on_validation": (
        "tuned_model"
        if tuned_val_metrics["f1"] > baseline_metrics["f1"]
        else "baseline"
    ),
    "decision": (
        "tuned model is slightly better on validation, keep it as current leader"
        if tuned_val_metrics["f1"] > baseline_metrics["f1"]
        else "keep baseline"
    ),
}
(tuning_dir / "comparison_vs_baseline.json").write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved files:")
for p in sorted(tuning_dir.iterdir()):
    print("-", p.name)

Saved files:
- best_params.json
- comparison_vs_baseline.json
- model.joblib
- val_metrics.json


In [10]:
param_grid_v2 = {
    "vectorizer__max_features": [32000, 35000, 38000],
    "classifier__C": [1.5, 2.0, 3.0],
}
grid_v2 = GridSearchCV(
    estimator=model,
    param_grid=param_grid_v2,
    scoring=make_scorer(f1_score),
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid_v2.fit(x_train, y_train)

print("Best params v2:", grid_v2.best_params_)
print("Best CV F1 v2:", grid_v2.best_score_)

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best params v2: {'classifier__C': 3.0, 'vectorizer__max_features': 38000}
Best CV F1 v2: 0.8973565822794781


In [11]:
best_model_v2 = grid_v2.best_estimator_
y_val_pred_v2 = best_model_v2.predict(x_val)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_val,
    y_val_pred_v2,
    average="binary",
    pos_label=1,
    zero_division=0,
)

tuned_v2_val_metrics = {
    "accuracy": float(accuracy_score(y_val, y_val_pred_v2)),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
}

print("Baseline:", baseline_metrics)
print("Tuned v1:", tuned_val_metrics)
print("Tuned v2:", tuned_v2_val_metrics)
print("Best params v2:", grid_v2.best_params_)

Baseline: {'accuracy': 0.8976, 'precision': 0.8994391025641025, 'recall': 0.8958499600957701, 'f1': 0.897640943622551}
Tuned v1: {'accuracy': 0.8994, 'precision': 0.8945063694267515, 'recall': 0.9042253521126761, 'f1': 0.8993396037622573}
Tuned v2: {'accuracy': 0.9022, 'precision': 0.897609561752988, 'recall': 0.9066398390342052, 'f1': 0.9021021021021021}
Best params v2: {'classifier__C': 3.0, 'vectorizer__max_features': 38000}


In [12]:
tuning_dir_v2 = Path("artifacts/tuning/tfidf_narrow_v2")
tuning_dir_v2.mkdir(parents=True, exist_ok=True)

# 1. сохраняем лучшую модель
joblib.dump(best_model_v2, tuning_dir_v2 / "model.joblib")

# 2. сохраняем лучшие параметры и CV score
best_params_payload_v2 = {
    "best_params": grid_v2.best_params_,
    "best_cv_f1": float(grid_v2.best_score_),
}
(tuning_dir_v2 / "best_params.json").write_text(
    json.dumps(best_params_payload_v2, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# 3. сохраняем validation metrics
(tuning_dir_v2 / "val_metrics.json").write_text(
    json.dumps(tuned_v2_val_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# 4. сравнение с baseline
comparison_v2 = {
    "baseline": baseline_metrics,
    "tuned_v1": tuned_val_metrics,
    "tuned_v2": tuned_v2_val_metrics,
    "best_params": grid_v2.best_params_,
    "delta_tuned_v2_minus_baseline": {
        metric: round(tuned_v2_val_metrics[metric] - baseline_metrics[metric], 6)
        for metric in baseline_metrics.keys()
    },
    "winner_on_validation": (
        "tuned_v2"
        if tuned_v2_val_metrics["f1"] > baseline_metrics["f1"]
        else "baseline"
    ),
    "decision": (
        "tuned_v2 is the current leader on validation"
        if tuned_v2_val_metrics["f1"] > baseline_metrics["f1"]
        else "keep baseline"
    ),
}
(tuning_dir_v2 / "comparison_vs_baseline.json").write_text(
    json.dumps(comparison_v2, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved files:")
for p in sorted(tuning_dir_v2.iterdir()):
    print("-", p.name)

Saved files:
- best_params.json
- comparison_vs_baseline.json
- model.joblib
- val_metrics.json


In [14]:
src_dir = Path("artifacts/tuning/tfidf_narrow_v2")
final_dir = Path("artifacts/final_baseline/tfidf_tuned_v2")
final_dir.mkdir(parents=True, exist_ok=True)

for filename in [
    "model.joblib",
    "val_metrics.json",
    "best_params.json",
    "comparison_vs_baseline.json",
]:
    shutil.copy2(src_dir / filename, final_dir / filename)

best_params = json.loads((src_dir / "best_params.json").read_text(encoding="utf-8"))["best_params"]

max_features = best_params["vectorizer__max_features"]
c_value = best_params["classifier__C"]

config_path = Path("configs/final_baseline_tfidf_tuned_v2.yaml")
config_text = textwrap.dedent(f"""
experiment:
  family: tfidf
  name: final_baseline_tuned_v2

seed: 42

paths:
  model_output: artifacts/final_baseline/tfidf_tuned_v2/model.joblib
  val_metrics_output: artifacts/final_baseline/tfidf_tuned_v2/val_metrics.json
  test_metrics_output: artifacts/final_baseline/tfidf_tuned_v2/test_metrics.json

model:
  type: logistic_regression
  max_features: {max_features}
  ngram_range: [1, 2]
  max_iter: 1000
""").strip() + "\n"

config_path.write_text(config_text, encoding="utf-8")

summary = {
    "status": "promoted_to_final_baseline",
    "source": "artifacts/tuning/tfidf_narrow_v2",
    "best_params": best_params,
}
(final_dir / "promotion_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Final baseline saved to:", final_dir)
print("Config created:", config_path)
print("\nFiles in final baseline:")
for p in sorted(final_dir.iterdir()):
    print("-", p.name)
print("\nNew baseline config:")
print(config_path.read_text(encoding="utf-8"))
print(f"\nChosen LogisticRegression C from tuning: {c_value}")

Final baseline saved to: artifacts/final_baseline/tfidf_tuned_v2
Config created: configs/final_baseline_tfidf_tuned_v2.yaml

Files in final baseline:
- best_params.json
- comparison_vs_baseline.json
- model.joblib
- promotion_summary.json
- val_metrics.json

New baseline config:
experiment:
  family: tfidf
  name: final_baseline_tuned_v2

seed: 42

paths:
  model_output: artifacts/final_baseline/tfidf_tuned_v2/model.joblib
  val_metrics_output: artifacts/final_baseline/tfidf_tuned_v2/val_metrics.json
  test_metrics_output: artifacts/final_baseline/tfidf_tuned_v2/test_metrics.json

model:
  type: logistic_regression
  max_features: 38000
  ngram_range: [1, 2]
  max_iter: 1000


Chosen LogisticRegression C from tuning: 3.0


In [16]:
package_dir = Path("download_package_tfidf_tuned_v2")
package_dir.mkdir(parents=True, exist_ok=True)

src_artifacts = Path("artifacts/final_baseline/tfidf_tuned_v2")
dst_artifacts = package_dir / "tfidf_tuned_v2"
shutil.copytree(src_artifacts, dst_artifacts, dirs_exist_ok=True)

src_config = Path("configs/final_baseline_tfidf_tuned_v2.yaml")
dst_config = package_dir / "final_baseline_tfidf_tuned_v2.yaml"
shutil.copy2(src_config, dst_config)

archive_path = shutil.make_archive("tfidf_tuned_v2_final", "zip", package_dir)

print("Archive created:", archive_path)
files.download(archive_path)

Archive created: /content/imdb-sentiment/tfidf_tuned_v2_final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>